# Period doubling and Feigenbaum drift in the logistic map

This notebook is the slower companion to `reports/period-doubling-and-feigenbaum.md`.

The narrow question is the useful one: **how do stable periods actually show up on a direct parameter scan, and how quickly do the superstable gap ratios start leaning toward Feigenbaum's constant?**


## 1. Why scan periods directly

The bifurcation diagram already shows the cascade visually, but it does not count stable periods explicitly.
A direct scan does something simpler:

1. warm up the orbit for one value of `r`
2. read the tail
3. ask for the smallest repeat period up to a fixed cutoff

That does not prove the whole interval has that period. It does give a clean public summary of which stable windows are still visible on the scan grid.


In [ ]:
from logisticlab.core import classify_parameter, feigenbaum_estimates, scan_periods, scan_superstable_doubling

rows = scan_periods(3.0, 4.0, samples=300, warmup=2400, keep=256, max_period=64)
counts = {}
chaos = 0
for row in rows:
    if row.detected_period is None:
        chaos += 1
    else:
        counts[row.detected_period] = counts.get(row.detected_period, 0) + 1
counts, chaos


The exact counts depend on the grid density and the scan cutoff, but the structural picture is stable:

- low periods occupy wide windows
- higher periods appear in thinner and thinner bands
- chaos takes over more of the axis as the cascade compresses


## 2. Superstable points give cleaner markers

At the center of a stable band, the critical point `x = 0.5` comes back to itself after one full cycle.
Those parameter values are easier to compare than the full window widths, because they turn the band centers into one explicit sequence.


In [ ]:
points = scan_superstable_doubling(6)
[(row.period, round(row.r, 12)) for row in points]


These values bunch up quickly near the chaos onset.
That is the geometric core of the Feigenbaum story: the period-doubling windows do not just get smaller, they shrink by a ratio that stabilizes numerically.


In [ ]:
estimates = feigenbaum_estimates(points)
[(row.period, round(row.delta, 6)) for row in estimates]


## 3. What the gap ratios are and are not saying

If `r_2, r_4, r_8, ...` are the superstable points, then the usual ratio estimate is

$$\delta_n = \frac{r_{2^{n-1}} - r_{2^{n-2}}}{r_{2^n} - r_{2^{n-1}}}. $$

In a finite public calculation, these estimates are not meant to be a high-precision constant hunt.
They are there to show that the scaling story is already visible with modest code and no heavy numerical stack.


## 4. Caveats

This notebook is intentionally bounded.

- the period scan depends on a finite tail length and a repeat cutoff
- unresolved points are not proofs of true chaos; some are just awkward for the current budget
- the superstable search is numerical bracketing, not symbolic exact algebra
- the Feigenbaum estimates here are educational approximations, not a maximal-precision computation


## 5. Problems and references

Try these next:

1. rerun the period scan with a tighter tolerance and see which thin windows disappear first
2. compare the direct period scan against the Lyapunov sweep to see where they agree and where the finite cutoff blurs the picture
3. extend the superstable search one or two more steps and ask when the public calculation stops being numerically honest

Useful references:

- the repo's own `reports/period-doubling-and-feigenbaum.md`
- the generated artifact `assets/period-doubling-atlas.svg`
- any standard chaos or nonlinear dynamics text covering Feigenbaum scaling in unimodal maps
